In [1]:
import gc
import os

import numpy as np
import pandas as pd
import torch
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader

from benchmarks import LightningModule, MultiTargetScaler, PrintEpochMetricsCallback
from dataset.compas import COMPAS1DDataset
from gcnn.base_model import GroupEquivariantCNN
from pac import PAC, smiles_to_pac
from utils import calc_num_rings

torch.set_float32_matmul_precision("medium")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

k_folds = 5
dataset_file = "../data/compas1/compas-1D_b3lyp_def2-svp.csv"
model_logs_dir = "../logs"
model_path_template = "../models/gcnn_3.7M-pac_grid-[COMPAS-1D]-fold_{fold}"
model_logs_template = "gcnn_3.7M-pac_grid-[COMPAS-1D]-fold_{fold}"

target_properties = ["HOMO_eV", "LUMO_eV", "GAP_eV"]
property_types = ["intensive", "intensive", "intensive"]

df = pd.read_csv(dataset_file)
df = df.dropna(subset=target_properties).reset_index(drop=True)

print(f"Total rows before parsing: {len(df)}")

smiles_list = df["smiles"].tolist()

parsed_pacs: list[PAC | None] = []
for smiles in smiles_list:
    try:
        parsed_pacs.append(smiles_to_pac(smiles))
    except Exception:
        parsed_pacs.append(None)

df["pac"] = pd.Series(parsed_pacs, index=df.index)

invalid_mask = df["pac"].isnull()
invalid_df = df[invalid_mask]
invalid_count = len(invalid_df)

if invalid_count > 0:
    print(f"\n{invalid_count} SMILES failed to parse and will be dropped:")
    for idx, row in invalid_df.iterrows():
        print(f" + Index {idx}: {row['smiles']}")

df = df[~invalid_mask].reset_index(drop=True)
print(f"Filtered out {invalid_count} invalid SMILES. Remaining valid rows: {len(df)}")

df["ring_count"] = df["smiles"].apply(calc_num_rings)

counts = df["ring_count"].value_counts()
rare_classes = counts[counts < 5].index.tolist()

df["stratify_group"] = df["ring_count"].replace(rare_classes, "rare").astype(str)

if 0 < (df["stratify_group"] == "rare").sum() < 5:
    most_freq = str(counts.idxmax())
    df.loc[df["stratify_group"] == "rare", "stratify_group"] = most_freq

skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

W0717 15:49:06.360000 28468 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Using device: cuda
Total rows before parsing: 8678

1071 SMILES failed to parse and will be dropped:
 + Index 24: c1ccc2c(c1)ccc1ccc3ccc4ccc5ccccc5c4c3c12
 + Index 58: c1ccc2c(c1)ccc1ccc3ccc4c5ccccc5c5ccccc5c4c3c12
 + Index 59: c1ccc2c(c1)ccc1c2ccc2ccc3ccc4ccc5ccccc5c4c3c21
 + Index 100: c1ccc2c(c1)ccc1ccc3ccc4ccc5ccc6ccccc6c5c4c3c12
 + Index 101: c1ccc2cc3c(ccc4ccc5ccc6ccc7ccccc7c6c5c43)cc2c1
 + Index 179: c1ccc2c(c1)ccc1ccc3ccc4ccc5c6ccccc6c6ccccc6c5c4c3c12
 + Index 180: c1ccc2c(c1)ccc1c3ccc4ccc5ccccc5c4c3c3c4ccccc4ccc3c21
 + Index 191: c1ccc2c(c1)ccc1c3ccccc3c3c(ccc4ccc5ccc6ccccc6c5c43)c21
 + Index 209: c1ccc2cc3c(ccc4ccc5c6ccccc6c6ccc7ccccc7c6c5c43)cc2c1
 + Index 213: c1ccc2c(c1)ccc1c3ccc4ccc5ccc6ccccc6c5c4c3c3ccccc3c21
 + Index 290: c1ccc2c(c1)ccc1ccc3c4ccccc4c4c5ccccc5c5ccccc5c4c3c12
 + Index 308: c1ccc2cc3c(ccc4c5ccccc5c5ccc6ccc7ccccc7c6c5c34)cc2c1
 + Index 313: c1ccc2c(c1)ccc1ccc3ccc4c5ccccc5c5ccc6ccccc6c5c4c3c12
 + Index 324: c1ccc2c(c1)ccc1ccc3c4ccccc4c4ccc5ccc6ccccc6c5c4c3c1

In [2]:
fold_results = []

for fold, (train_val_idx, test_idx) in enumerate(
    skf.split(df, df["stratify_group"]), start=1
):
    print(f"\n======[ Fold {fold}/{k_folds} ]======")

    train_idx, val_idx = train_test_split(
        train_val_idx,
        test_size=0.125,
        random_state=42,
        stratify=df.loc[train_val_idx, "stratify_group"],
    )

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)

    train_dataset = COMPAS1DDataset(train_df, target_properties=target_properties)
    val_dataset = COMPAS1DDataset(val_df, target_properties=target_properties)

    scaler = MultiTargetScaler()
    scaler.fit(train_dataset.targets.numpy())

    train_dataset.targets = torch.tensor(
        scaler.transform(train_dataset.targets.numpy()), dtype=torch.float32
    )
    val_dataset.targets = torch.tensor(
        scaler.transform(val_dataset.targets.numpy()), dtype=torch.float32
    )

    train_loader = DataLoader(
        train_dataset, batch_size=256, shuffle=True, num_workers=4
    )
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=4)

    model = GroupEquivariantCNN(
        property_types=property_types, kernel_size=5, num_hidden=5, hidden_channels=48
    )
    lightning_model = LightningModule(model, target_properties, scaler, lr=1e-3)

    save_path_dir = model_path_template.format(fold=fold)
    os.makedirs(save_path_dir, exist_ok=True)

    checkpoint_callback = ModelCheckpoint(
        dirpath=save_path_dir,
        filename="best-{epoch}-{val_loss:.5f}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )

    logger = TensorBoardLogger(
        save_dir=model_logs_dir, name=model_logs_template.format(fold=fold)
    )

    trainer = pl.Trainer(
        max_epochs=100,
        gradient_clip_val=1.0,
        gradient_clip_algorithm="norm",
        accelerator="auto",
        logger=logger,
        devices=1,
        callbacks=[checkpoint_callback, PrintEpochMetricsCallback(target_properties)],
        enable_checkpointing=True,
        num_sanity_val_steps=0,
        log_every_n_steps=1,
        precision="bf16-mixed" if device.type == "cuda" else "32-true",
        enable_progress_bar=False,
    )

    trainer.fit(lightning_model, train_loader, val_loader)

    best_model_score = checkpoint_callback.best_model_score
    if best_model_score is None:
        raise RuntimeError("The checkpoint callback did not record a validation loss.")
    best_val_loss = best_model_score.item()
    print(f"Best scaled validation loss for fold {fold}: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)

    best_model_path = checkpoint_callback.best_model_path
    best_lightning_model = LightningModule.load_from_checkpoint(
        best_model_path,
        model=model,
        property_names=target_properties,
        scaler=scaler,
        enable_progress_bar=False,
    )
    torch.save(
        best_lightning_model.model.state_dict(),
        os.path.join(save_path_dir, "best.ckpt"),
    )

    lightning_model.cpu()
    best_lightning_model.cpu()

    del train_dataset, val_dataset, train_loader, val_loader
    del model, lightning_model, best_lightning_model
    del trainer, checkpoint_callback, logger

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Final k-fold results ]======")
print(f"Scaled Loss per fold: {[round(float(x), 4) for x in fold_results]}")
print(f"Average Scaled Loss: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")


======[ Fold 1/5 ]======
Generating PAC representations for 5324 molecules...
Generating PAC representations for 761 molecules...


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\model_summary\model_summary.py:242: Precis

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model      │ GroupEquivariantCNN │  3.7 M │ train │     0 │
│ 1 │ criterion  │ MSELoss             │      0 │ train │     0 │
│ 2 │ train_rmsd │ ModuleDict          │      0 │ train │     0 │
│ 3 │ val_rmsd   │ ModuleDict          │      0 │ train │     0 │
│ 4 │ test_rmsd  │ ModuleDict          │      0 │ train │     0 │
│ 5 │ test_mae   │ ModuleDict          │      0 │ train │     0 │
│ 6 │ test_r2    │ ModuleDict          │      0 │ train │     0 │
│ 7 │ test_max   │ ModuleDict          │      0 │ train │     0 │
│ 8 │ test_min   │ ModuleDict          │      0 │ train │     0 │
└───┴────────────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 3.7 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.7 M                                                                                                
Total estimated model params size (MB): 14.837                                                                     
Modules in train mode: 59                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.


...

Trainable params: 3.7 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.7 M                                                                                                
Total estimated model params size (MB): 14.837                                                                     
Modules in train mode: 59                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.


Epoch 001 | Scaled Train Loss: N/A | Scaled Val Loss: 2.64426 | HOMO_eV Val RMSD: 0.43612 | LUMO_eV Val RMSD: 0.23032 | GAP_eV Val RMSD: 0.48776
Epoch 002 | Scaled Train Loss: 0.18883 | Scaled Val Loss: 5.64401 | HOMO_eV Val RMSD: 0.61056 | LUMO_eV Val RMSD: 0.32105 | GAP_eV Val RMSD: 0.81445
Epoch 003 | Scaled Train Loss: 0.03756 | Scaled Val Loss: 9.96552 | HOMO_eV Val RMSD: 0.71126 | LUMO_eV Val RMSD: 0.59396 | GAP_eV Val RMSD: 1.18196
Epoch 004 | Scaled Train Loss: 0.02534 | Scaled Val Loss: 0.65512 | HOMO_eV Val RMSD: 0.11191 | LUMO_eV Val RMSD: 0.19808 | GAP_eV Val RMSD: 0.35559
Epoch 005 | Scaled Train Loss: 0.02005 | Scaled Val Loss: 0.05909 | HOMO_eV Val RMSD: 0.05339 | LUMO_eV Val RMSD: 0.04814 | GAP_eV Val RMSD: 0.05869
Epoch 006 | Scaled Train Loss: 0.01786 | Scaled Val Loss: 0.32891 | HOMO_eV Val RMSD: 0.10413 | LUMO_eV Val RMSD: 0.13169 | GAP_eV Val RMSD: 0.21586
Epoch 007 | Scaled Train Loss: 0.01637 | Scaled Val Loss: 0.04026 | HOMO_eV Val RMSD: 0.05220 | LUMO_eV Val RM

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 100 | Scaled Train Loss: 0.00526 | Scaled Val Loss: 0.00225 | HOMO_eV Val RMSD: 0.01692 | LUMO_eV Val RMSD: 0.01093 | GAP_eV Val RMSD: 0.01819
Best scaled validation loss for fold 5: 0.0020

======[ Final k-fold results ]======
Scaled Loss per fold: [0.0025, 0.0022, 0.0016, 0.0023, 0.002]
Average Scaled Loss: 0.0021 ± 0.0003


In [3]:
metric_keys = ["mae", "rmsd", "max", "min", "r2"]
all_results = {
    "train": {prop: {m: [] for m in metric_keys} for prop in target_properties},
    "test": {prop: {m: [] for m in metric_keys} for prop in target_properties},
}

for fold, (train_idx, test_idx) in enumerate(skf.split(df, df["ring_count"]), 1):
    print(f"\n======[ Evaluating fold {fold}/{k_folds} ]======")

    train_df = df.iloc[train_idx]
    test_df = df.iloc[test_idx]

    train_dataset = COMPAS1DDataset(train_df, target_properties=target_properties)
    test_dataset = COMPAS1DDataset(test_df, target_properties=target_properties)

    scaler = MultiTargetScaler()
    scaler.fit(train_dataset.targets.numpy())

    train_dataset.targets = torch.tensor(
        scaler.transform(train_dataset.targets.numpy()), dtype=torch.float32
    )
    test_dataset.targets = torch.tensor(
        scaler.transform(test_dataset.targets.numpy()), dtype=torch.float32
    )

    train_loader = DataLoader(
        train_dataset, batch_size=256, shuffle=False, num_workers=12
    )
    test_loader = DataLoader(
        test_dataset, batch_size=256, shuffle=False, num_workers=12
    )

    model = GroupEquivariantCNN(
        property_types=property_types, kernel_size=5, num_hidden=5, hidden_channels=48
    )
    load_path = model_path_template.format(fold=fold) + "/best.ckpt"
    model.load_state_dict(torch.load(load_path, map_location=device))

    lightning_model = LightningModule(model, target_properties, scaler)

    trainer = pl.Trainer(
        accelerator="auto", devices=1, logger=False, enable_progress_bar=False
    )

    print("\n======[ Train Set Evaluation ]======")
    train_results = trainer.test(
        lightning_model, dataloaders=train_loader, verbose=False
    )[0]

    print("\n======[ Test Set Evaluation ]======")
    test_results = trainer.test(
        lightning_model, dataloaders=test_loader, verbose=False
    )[0]

    for i, prop in enumerate(target_properties):
        print(f"======[ {prop} ]======")
        t_mae = train_results[f"test_mae_{prop}"]
        t_rmsd = train_results[f"test_rmsd_{prop}"]
        t_max = train_results[f"test_max_{prop}"]
        t_min = train_results[f"test_min_{prop}"]
        t_r2 = train_results[f"test_r2_{prop}"]
        print(
            f" + Train -> MAE: {t_mae:.4f}\tRMSD: {t_rmsd:.4f}\tMax: {t_max:.4f}\tMin: {t_min:.4f}\tR^2: {t_r2:.4f}"
        )

        te_mae = test_results[f"test_mae_{prop}"]
        te_rmsd = test_results[f"test_rmsd_{prop}"]
        te_max = test_results[f"test_max_{prop}"]
        te_min = test_results[f"test_min_{prop}"]
        te_r2 = test_results[f"test_r2_{prop}"]
        print(
            f" + Test -> MAE: {te_mae:.4f}\tRMSD: {te_rmsd:.4f}\tMax: {te_max:.4f}\tMin: {te_min:.4f}\tR^2: {te_r2:.4f}"
        )

        all_results["train"][prop]["mae"].append(t_mae)
        all_results["train"][prop]["rmsd"].append(t_rmsd)
        all_results["train"][prop]["max"].append(t_max)
        all_results["train"][prop]["min"].append(t_min)
        all_results["train"][prop]["r2"].append(t_r2)

        all_results["test"][prop]["mae"].append(te_mae)
        all_results["test"][prop]["rmsd"].append(te_rmsd)
        all_results["test"][prop]["max"].append(te_max)
        all_results["test"][prop]["min"].append(te_min)
        all_results["test"][prop]["r2"].append(te_r2)

    del model
    del lightning_model
    del trainer
    del scaler
    del train_results
    del test_results
    del train_loader
    del test_loader
    del train_dataset
    del test_dataset
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Cross-validation results (mean ± std) ]======")

for prop in target_properties:
    print(f"\n======[ {prop} ]======")

    t_stats = {
        m: (
            np.mean(all_results["train"][prop][m]),
            np.std(all_results["train"][prop][m], ddof=1),
        )
        for m in metric_keys
    }

    print(
        f" + Train -> "
        f"MAE: {t_stats['mae'][0]:.4f} ± {t_stats['mae'][1]:.4f}\t"
        f"RMSD: {t_stats['rmsd'][0]:.4f} ± {t_stats['rmsd'][1]:.4f}\t"
        f"Max: {t_stats['max'][0]:.4f} ± {t_stats['max'][1]:.4f}\t"
        f"Min: {t_stats['min'][0]:.4f} ± {t_stats['min'][1]:.4f}\t"
        f"R^2: {t_stats['r2'][0]:.4f} ± {t_stats['r2'][1]:.4f}"
    )

    te_stats = {
        m: (
            np.mean(all_results["test"][prop][m]),
            np.std(all_results["test"][prop][m], ddof=1),
        )
        for m in metric_keys
    }

    print(
        f" + Test -> "
        f"MAE: {te_stats['mae'][0]:.4f} ± {te_stats['mae'][1]:.4f}\t"
        f"RMSD: {te_stats['rmsd'][0]:.4f} ± {te_stats['rmsd'][1]:.4f}\t"
        f"Max: {te_stats['max'][0]:.4f} ± {te_stats['max'][1]:.4f}\t"
        f"Min: {te_stats['min'][0]:.4f} ± {te_stats['min'][1]:.4f}\t"
        f"R^2: {te_stats['r2'][0]:.4f} ± {te_stats['r2'][1]:.4f}"
    )

D:\Coding\pac-gcnn\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(



======[ Evaluating fold 1/5 ]======
Generating PAC representations for 6085 molecules...
Generating PAC representations for 1522 molecules...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'test_dataloader' to speed up the dataloader worker initialization.



======[ Train Set Evaluation ]======


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...


======[ Test Set Evaluation ]======
======[ HOMO_eV ]======
 + Train -> MAE: 0.0058	RMSD: 0.0085	Max: 0.1484	Min: 0.0000	R^2: 0.9980
 + Test -> MAE: 0.0056	RMSD: 0.0082	Max: 0.0631	Min: 0.0000	R^2: 0.9980
======[ LUMO_eV ]======
 + Train -> MAE: 0.0063	RMSD: 0.0090	Max: 0.1828	Min: 0.0000	R^2: 0.9984
 + Test -> MAE: 0.0060	RMSD: 0.0083	Max: 0.0655	Min: 0.0000	R^2: 0.9985
======[ GAP_eV ]======
 + Train -> MAE: 0.0097	RMSD: 0.0144	Max: 0.4192	Min: 0.0000	R^2: 0.9988
 + Test -> MAE: 0.0094	RMSD: 0.0127	Max: 0.0867	Min: 0.0000	R^2: 0.9990

======[ Evaluating fold 5/5 ]======
Generating PAC representations for 6086 molecules...
Generating PAC representations for 1521 molecules...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



======[ Train Set Evaluation ]======


D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'test_dataloader' to speed up the dataloader worker initialization.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



======[ Test Set Evaluation ]======
======[ HOMO_eV ]======
 + Train -> MAE: 0.0047	RMSD: 0.0074	Max: 0.1104	Min: 0.0000	R^2: 0.9985
 + Test -> MAE: 0.0049	RMSD: 0.0085	Max: 0.1378	Min: 0.0000	R^2: 0.9980
======[ LUMO_eV ]======
 + Train -> MAE: 0.0053	RMSD: 0.0081	Max: 0.1312	Min: 0.0000	R^2: 0.9987
 + Test -> MAE: 0.0056	RMSD: 0.0090	Max: 0.1012	Min: 0.0000	R^2: 0.9984
======[ GAP_eV ]======
 + Train -> MAE: 0.0080	RMSD: 0.0123	Max: 0.2537	Min: 0.0000	R^2: 0.9991
 + Test -> MAE: 0.0085	RMSD: 0.0143	Max: 0.2067	Min: 0.0000	R^2: 0.9988

======[ Cross-validation results (mean ± std) ]======

======[ HOMO_eV ]======
 + Train -> MAE: 0.0051 ± 0.0006	RMSD: 0.0090 ± 0.0022	Max: 0.2552 ± 0.3333	Min: 0.0000 ± 0.0000	R^2: 0.9977 ± 0.0013
 + Test -> MAE: 0.0051 ± 0.0005	RMSD: 0.0082 ± 0.0010	Max: 0.1097 ± 0.0625	Min: 0.0000 ± 0.0000	R^2: 0.9981 ± 0.0005

======[ LUMO_eV ]======
 + Train -> MAE: 0.0055 ± 0.0009	RMSD: 0.0095 ± 0.0021	Max: 0.2995 ± 0.3488	Min: 0.0000 ± 0.0000	R^2: 0.9981 ± 0.0010